In [16]:
# Task 1
import numpy as np

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ", "9", "12", "", "45", "182", "96"]

def parse_int(value):
    if value is None:
        return None
    try:
        return int(str(value).strip())
    except ValueError:
        return None

# Clean values and track which months failed conversion
parsed = [parse_int(val) for val in machakos_raw]
imputed_months = [months[i] for i, val in enumerate(parsed) if val is None]

# Compute rounded mean of valid entries and impute missing values
valid_values = [val for val in parsed if val is not None]
mean_rainfall = round(sum(valid_values) / len(valid_values))
machakos_clean = [mean_rainfall if val is None else val for val in parsed]

print("Imputed months:", imputed_months)
print("Imputed value:", mean_rainfall)
print("Cleaned Machakos row:", machakos_clean)

Imputed months: ['Apr', 'Sep']
Imputed value: 68
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [17]:
#Task 2
# Combine the four clean rows and the cleaned Machakos row
all_counties_data = clean_rows + [machakos_clean]

# Convert to a 2D NumPy array
rainfall_data = np.array(all_counties_data)

# Confirm shape and dtype
print("Shape of rainfall data array:", rainfall_data.shape)
print("Data type of rainfall data array:", rainfall_data.dtype)

Shape of rainfall data array: (5, 12)
Data type of rainfall data array: int64


In [18]:
# Task 3
annual_totals = np.sum(rainfall_data, axis=1)
monthly_averages = np.mean(rainfall_data, axis=0)

print("Annual total rainfall per county:")
for county, total in zip(counties, annual_totals):
    print(f"  {county}: {total} mm")

print("\nAverage monthly rainfall across all counties:")
for month, avg in zip(months, monthly_averages):
    print(f"  {month}: {avg:.1f} mm")

Annual total rainfall per county:
  Nairobi: 1045 mm
  Nakuru: 916 mm
  Kisumu: 1371 mm
  Mombasa: 1075 mm
  Machakos: 811 mm

Average monthly rainfall across all counties:
  Jan: 47.0 mm
  Feb: 47.6 mm
  Mar: 91.0 mm
  Apr: 164.4 mm
  May: 171.2 mm
  Jun: 67.6 mm
  Jul: 56.6 mm
  Aug: 58.2 mm
  Sep: 64.2 mm
  Oct: 67.8 mm
  Nov: 127.4 mm
  Dec: 80.6 mm


In [19]:
# Task 4
wettest_indices = np.argmax(rainfall_data, axis=1)
wettest_months = [months[idx] for idx in wettest_indices]

print("Wettest month per county:")
for county, month in zip(counties, wettest_months):
    print(f"  {county}: {month}")

Wettest month per county:
  Nairobi: Apr
  Nakuru: Apr
  Kisumu: Apr
  Mombasa: May
  Machakos: Nov


In [20]:
# Task 5
dry_mask = rainfall_data < 50
dry_month_counts = np.sum(dry_mask, axis=1)

print("Dry months (< 50 mm) per county:")
for county, count in zip(counties, dry_month_counts):
    print(f"  {county}: {count} months")

Dry months (< 50 mm) per county:
  Nairobi: 4 months
  Nakuru: 3 months
  Kisumu: 0 months
  Mombasa: 2 months
  Machakos: 6 months


In [21]:
# Task 6
two_month_sums = rainfall_data[:, :-1] + rainfall_data[:, 1:]
best_window_indices = np.argmax(two_month_sums, axis=1)

planting_windows = [f"{months[idx]}-{months[idx + 1]}" for idx in best_window_indices]

print("Recommended planting window per county:")
for county, window, idx in zip(counties, planting_windows, best_window_indices):
    print(f"  {county}: {window} ({two_month_sums[counties.index(county), idx]} mm)")

Recommended planting window per county:
  Nairobi: Apr-May (395 mm)
  Nakuru: Apr-May (265 mm)
  Kisumu: Apr-May (420 mm)
  Mombasa: Apr-May (390 mm)
  Machakos: Nov-Dec (278 mm)


In [22]:
import csv

output_file = "rainfall_summary.csv"
headers = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open(output_file, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    for row in zip(counties, annual_totals, wettest_months, dry_month_counts, planting_windows):
        writer.writerow(row)

# Verify the file contents
with open(output_file, mode="r", encoding="utf-8") as f:
    print(f.read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,Apr,4,Apr-May
Nakuru,916,Apr,3,Apr-May
Kisumu,1371,Apr,0,Apr-May
Mombasa,1075,May,2,Apr-May
Machakos,811,Nov,6,Nov-Dec



## Planting Advisory

* **Nairobi, Nakuru, Kisumu, and Mombasa (April–May Window):** Farmers in these four counties should prepare land in March and plant at the onset of the **April–May** long rains, which deliver the highest consecutive two-month rainfall (265 mm in Nakuru up to 420 mm in Kisumu).
* **Machakos (November–December Window):** Unlike the other counties, Machakos peaks late in the year; farmers should prioritize planting during the **November–December** short rains (278 mm combined) and use drought-tolerant crops during its six dry months (< 50 mm).
* **Dry-Spell Management:** Kisumu experiences zero dry months and supports year-round cropping, whereas Nairobi (4 dry months) and Machakos (6 dry months) require water harvesting or mulching between June and September.

## Data Quality Notes

* **Manual Entry Cleaning:** The `machakos_raw` dataset was typed by hand and contained non-numeric entries (`"N/A"` in April and an empty string `""` in September) alongside whitespace-padded numbers (`" 20 "`). These were parsed with a `try/except ValueError` function.
* **Mean Imputation:** The two missing Machakos readings (**Apr** and **Sep**) were replaced with `68` mm, the rounded average of the 10 valid Machakos monthly readings.
* **Analytical Caveat:** Because `68` mm sits above the 50 mm dry-month threshold, imputing April and September with the mean may slightly underestimate Machakos's true dry-month count and flatten seasonal extremes compared to actual weather-station records.